In [ ]:
#!/usr/bin/env python3
"""Phase 1 — Baseline LSTM, multi-rate, on the 720 h `systematic-720` data.

    conda run -n exadigit python run_phase1.py


ARCHITECTURE (multi-rate variant, `MultiRateLSTM`)
    One LSTM + temporal-attention encoder per branch over concat(u_hist,
    y_hist) at that branch's own rate, the branch contexts concatenated, then
    a direct per-head MLP decoder. No operator basis and no trunk — that is
    exactly what separates phase 1 from phases 2+, and keeping it is the point
    of running it as a baseline.

WHAT THIS RUN IS FOR
    The floor of the comparison table. Everything phases 2-6 add (operator
    basis, Fourier trunk, domain experts, federated heads, physics) has to
    earn its place against this.
"""


In [ ]:
# 1  Imports, seed, run registry (bump RUN_ID every run; NEVER reuse one)
import os, sys, json, time, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt

# Use THIS repo's fmu2ml (<repo>/surrogate_models/..), whatever the kernel's
# working directory, PYTHONPATH, or an earlier import in this kernel say.
_nb_file = globals().get("__vsc_ipynb_file__")      # notebook path under VS Code
NB_DIR = os.path.dirname(os.path.abspath(_nb_file)) if _nb_file else os.getcwd()
REPO_ROOT = os.path.dirname(NB_DIR)
os.chdir(NB_DIR)                                    # run outputs -> surrogate_models/runs/
for _m in [m for m in sys.modules if m == "fmu2ml" or m.startswith("fmu2ml.")]:
    del sys.modules[_m]                             # forget an fmu2ml imported from elsewhere
sys.path.insert(0, REPO_ROOT)
import fmu2ml.surrogate as surrogate
print("fmu2ml:", os.path.dirname(surrogate.__file__))
assert surrogate.__file__.startswith(os.path.join(REPO_ROOT, "fmu2ml", "")), (
    f"fmu2ml came from {surrogate.__file__}, expected {REPO_ROOT}/fmu2ml "
    f"(the repo above {NB_DIR}; run the kernel from this notebook's folder)")


In [ ]:

from fmu2ml.surrogate.phase_common import (PhaseRun, task_kwargs,
                                          MAX_EPOCHS, PATIENCE, WEIGHT_DECAY, scaled_lr)
import fmu2ml.surrogate as surrogate

RUN_ID = "phase1_lstm_mr"

RUN_NOTES = (
    "Phase 1 baseline LSTM trained MULTI-RATE on the 720 h "
)

config = surrogate.Phase1Config(
    **task_kwargs(),

    #  phase-1 architecture ────────────────────────────────────────────────
    lstm_hidden_size=256,
    lstm_num_layers=2,
    lstm_dropout=0.1,
    attention_heads=4,
    attention_dropout=0.1,
    decoder_hidden_sizes=[512, 256],
    decoder_dropout=0.1,

    #  training ────────────────────────────────────────────────────────────
    # 1e-3 is this phase's own default and is appropriate for a from-scratch
    # LSTM; MR-MIONet's 3e-5 is tuned for its much deeper operator stack and
    # would badly underfit here. Each phase trains at ITS OWN best-known
    # settings — the task is what must be identical, not the hyperparameters.
    learning_rate=scaled_lr(1e-3),
    # weight decay, epoch budget and patience are shared by every phase
    # (see phase_common); these are the values the reported runs used.
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)

if __name__ == "__main__":
    PhaseRun(run_id=RUN_ID, arch="lstm", phase="1",
             config=config, notes=RUN_NOTES).execute()
